# aiohttp_retry

## Example 1: a 503 is retried until the server answers 200

In [ ]:
# Example 1: the server fails twice with 503, then answers; ExponentialRetry retries it and returns the third answer
from aiohttp import web
from aiohttp_retry import ExponentialRetry, RetryClient

hits = []
async def flaky(request):
    hits.append(request.path)
    return web.Response(status=503) if len(hits) < 3 else web.json_response({"ok": True})

app = web.Application(); app.router.add_get("/flaky", flaky)
runner = web.AppRunner(app); await runner.setup()
site = web.TCPSite(runner, "127.0.0.1", 0); await site.start()
port = runner.addresses[0][1]
options = ExponentialRetry(attempts=5, start_timeout=0.01, statuses={503})
async with RetryClient(retry_options=options) as client:
    async with client.get(f"http://127.0.0.1:{port}/flaky") as response:
        body = await response.json()
await runner.cleanup()
assert response.status == 200 and body == {"ok": True} and len(hits) == 3, (response.status, body, hits)
print("Example 1: status", response.status, "after", len(hits), "attempts")

## Example 2: after the last attempt the last response is returned as it is

In [ ]:
# Example 2: a server that always answers 500 is tried `attempts` times; the client hands back the final 500
from aiohttp import web
from aiohttp_retry import ListRetry, RetryClient

count = {"n": 0}
async def broken(request):
    count["n"] += 1
    return web.Response(status=500, text="still broken")

app = web.Application(); app.router.add_get("/broken", broken)
runner = web.AppRunner(app); await runner.setup()
site = web.TCPSite(runner, "127.0.0.1", 0); await site.start()
port = runner.addresses[0][1]
async with RetryClient(retry_options=ListRetry(timeouts=[0.01, 0.01, 0.01])) as client:
    async with client.get(f"http://127.0.0.1:{port}/broken") as response:
        text = await response.text()
await runner.cleanup()
assert response.status == 500 and text == "still broken" and count["n"] == 3, (response.status, count)
print("Example 2: gave up after", count["n"], "attempts with status", response.status)

## Example 3: a different request on each attempt (`RequestParams`)

In [ ]:
# Example 3: client.requests() walks a list of RequestParams, one per attempt: a dead path first, then a live one
from aiohttp import web
from aiohttp_retry import RequestParams, RetryClient, ExponentialRetry

async def ok(request):
    return web.Response(text=f"served {request.path}")

app = web.Application(); app.router.add_get("/live", ok)
runner = web.AppRunner(app); await runner.setup()
site = web.TCPSite(runner, "127.0.0.1", 0); await site.start()
base = f"http://127.0.0.1:{runner.addresses[0][1]}"
plan = [RequestParams("GET", f"{base}/dead"), RequestParams("GET", f"{base}/live")]
async with RetryClient(retry_options=ExponentialRetry(attempts=2, start_timeout=0.01, statuses={404})) as client:
    async with client.requests(params_list=plan) as response:
        text = await response.text()
await runner.cleanup()
assert response.status == 200 and text == "served /live", (response.status, text)
print("Example 3:", text, "on the second attempt")

## Example 4: a callback decides whether a 200 is good enough

In [ ]:
# Example 4: evaluate_response_callback returns False (retry) until the JSON body says the job is done
from aiohttp import web
from aiohttp_retry import ExponentialRetry, RetryClient

states = iter(["running", "running", "done"])
async def job(request):
    return web.json_response({"state": next(states)})

async def finished(response):
    return (await response.json())["state"] == "done"

app = web.Application(); app.router.add_get("/job", job)
runner = web.AppRunner(app); await runner.setup()
site = web.TCPSite(runner, "127.0.0.1", 0); await site.start()
options = ExponentialRetry(attempts=4, start_timeout=0.01, evaluate_response_callback=finished)
async with RetryClient(retry_options=options) as client:
    async with client.get(f"http://127.0.0.1:{runner.addresses[0][1]}/job") as response:
        state = (await response.json())["state"]
await runner.cleanup()
assert state == "done", state
print("Example 4: job state", state)

## Example 5: the wait schedules of the retry option classes

In [ ]:
# Example 5: get_timeout(attempt) for each schedule: exponential with a cap, a fixed list, Fibonacci, bounded random
from aiohttp_retry import ExponentialRetry, FibonacciRetry, ListRetry, RandomRetry

exp = ExponentialRetry(attempts=5, start_timeout=0.1, factor=2.0, max_timeout=0.5)
assert [round(exp.get_timeout(a), 3) for a in range(5)] == [0.1, 0.2, 0.4, 0.5, 0.5]  # start * factor**attempt, capped
listed = ListRetry(timeouts=[1.0, 2.0, 3.0])
assert [listed.get_timeout(a) for a in range(3)] == [1.0, 2.0, 3.0] and listed.attempts == 3
fib = FibonacciRetry(attempts=5, multiplier=1.0, max_timeout=10.0)
fib_waits = [fib.get_timeout(a) for a in range(1, 6)]
assert fib_waits == [2.0, 3.0, 5.0, 8.0, 10.0], fib_waits  # 1, 1, 2, 3, 5, 8, 13 ... capped at max_timeout
rnd = RandomRetry(attempts=3, min_timeout=0.2, max_timeout=0.4)
assert all(0.2 <= rnd.get_timeout(1) <= 0.4 for _ in range(100))
print("Example 5: exponential", [round(exp.get_timeout(a), 3) for a in range(5)], "| fibonacci", fib_waits)